In [7]:
#라이브러리 호출
import numpy as np
import pandas as p
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV

In [8]:
#데이터 불러오기
df = pd.read_csv('kaggle_data.csv')
df.head(5)

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0


In [9]:
#데이터 파악하기
def simple_summary(df):
    print("\n" + "="*50)
    print(f"       [ DATASET SUMMARY ]")
    print("="*50)
    print(f"▶ Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("-"*50)
    print(f"{'Column Name':<20} | {'Dtype':<10} | {'Nulls'}")
    print("-"*50)

    for col in df.columns:
        null_count = df[col].isnull().sum()
        dtype = str(df[col].dtype)
        # 결측치가 있으면 강조(★) 표시
        null_str = f"{null_count} (!!)" if null_count > 0 else f"{null_count}"
        print(f"{col[:20]:<20} | {dtype:<10} | {null_str}")

    print("="*50 + "\n")

# 사용 예시
simple_summary(df)


       [ DATASET SUMMARY ]
▶ Shape: 230130 rows, 6 columns
--------------------------------------------------
Column Name          | Dtype      | Nulls
--------------------------------------------------
id                   | int64      | 0
date                 | object     | 0
country              | object     | 0
store                | object     | 0
product              | object     | 0
num_sold             | float64    | 8871 (!!)



In [10]:
categorical_cols = ['country', 'store', 'product','date']

# 2. LabelEncoder 적용
le = LabelEncoder()

for col in categorical_cols:
    # 각 컬럼별로 학습 및 변환
    df[col] = le.fit_transform(df[col])
    print(f"✅ {col:<10} 변환 완료")

print("\n" + "="*30)
print(df[categorical_cols].dtypes)

✅ country    변환 완료
✅ store      변환 완료
✅ product    변환 완료
✅ date       변환 완료

country    int64
store      int64
product    int64
date       int64
dtype: object


In [11]:
df = df.dropna(subset=['num_sold'])

X=df.drop(["num_sold"],axis=1)
y=df["num_sold"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=33)

In [13]:
rf = RandomForestRegressor(random_state=33, n_jobs=-1)

# 2. 하이퍼파라미터 그리드 (학습 속도를 위해 가볍게 설정)
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, 15],
    'min_samples_split': [2, 4]
}

# 3. GridSearchCV 설정 (MAPE 기준)
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_percentage_error', # MAPE 기준
    verbose=2
)

# 4. 학습 시작
print("🚀 MAPE 기준으로 GridSearchCV 최적화 시작...")
grid_search.fit(X_train, y_train)

# 5. 예측 및 최종 MAPE 계산
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

# 실제 MAPE (0.1이면 10% 오차를 의미)
final_mape = mean_absolute_percentage_error(y_test, y_pred)

print("\n" + "="*50)
print(f"✅ 최적 파라미터: {grid_search.best_params_}")
print(f"📊 최종 검증 MAPE: {final_mape:.4f} ({final_mape*100:.2f}%)")
print("="*50)

🚀 MAPE 기준으로 GridSearchCV 최적화 시작...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_depth=15, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=15, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=15, min_samples_split=2, n_estimators=100; total time=   0.1s
[CV] END max_depth=15, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_depth=15, min_samples_split=4, n_estimators=100; total time=   0.1s
[CV] END max_d

ValueError: 
All the 12 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py", line 360, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py", line 2961, in validate_data
    X, y = check_X_y(X, y, **check_param

In [12]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error

# 데이터 불러오기
df = pd.read_csv("kaggle_data.csv")

# 타깃 결측치 제거
df = df.dropna(subset=["num_sold"]).copy()

# 날짜 변환
df["date"] = pd.to_datetime(df["date"])

# 날짜 특성 생성
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek
df["dayofyear"] = df["date"].dt.dayofyear
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

# 장기적인 판매량 변화
df["trend"] = (df["date"] - df["date"].min()).dt.days

# 연간 계절성
df["sin_dayofyear"] = np.sin(
    2 * np.pi * df["dayofyear"] / 365.25
)
df["cos_dayofyear"] = np.cos(
    2 * np.pi * df["dayofyear"] / 365.25
)

# 주간 계절성
df["sin_dayofweek"] = np.sin(
    2 * np.pi * df["dayofweek"] / 7
)
df["cos_dayofweek"] = np.cos(
    2 * np.pi * df["dayofweek"] / 7
)

# 범주형 변수 지정
categorical_cols = ["country", "store", "product"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

# id와 원본 date는 제외
feature_cols = [
    "country",
    "store",
    "product",
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "trend",
    "sin_dayofyear",
    "cos_dayofyear",
    "sin_dayofweek",
    "cos_dayofweek"
]

X = df[feature_cols]
y = df["num_sold"]

# 기존 결과와 비교하기 위한 동일한 랜덤 분할
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=33
)

# LightGBM 모델
model = LGBMRegressor(
    objective="regression",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=30,
    reg_lambda=1.0,
    random_state=33,
    n_jobs=-1,
    verbosity=-1
)

# 타깃을 로그 변환하여 학습
model.fit(
    X_train,
    np.log1p(y_train),
    categorical_feature=categorical_cols
)

# 예측 후 원래 단위로 복원
log_pred = model.predict(X_test)
y_pred = np.expm1(log_pred)

# 음수 또는 0 예측 방지
y_pred = np.clip(y_pred, 1, None)

# MAPE 계산
final_mape = mean_absolute_percentage_error(y_test, y_pred)

print("=" * 50)
print(f"최종 검증 MAPE: {final_mape:.4f}")
print(f"백분율 기준 MAPE: {final_mape * 100:.2f}%")
print("=" * 50)

최종 검증 MAPE: 0.0435
백분율 기준 MAPE: 4.35%
